# NB09c: Stacking + LightGBM + NaN Ablation + Classical ML (v9)

> Copyright (C) 2024-2026 Marco Heinzen - SPDX-License-Identifier: AGPL-3.0-or-later
> Part of the Master Thesis "Building Damage Assessment with Multimodal Satellite Time Series and Machine Learning in the Russia-Ukraine War 2022-2026"
> Code hosted at https://github.com/marcoheinzen/bda
> Parts of this code were written or improved with the assistance of Claude (Anthropic); all other code, and the concept, research, architecture, design, execution, testing and validation throughout, are the author's work.


## Core experiments:
- E1: Stacking meta-learner (LogReg on RF+GBM+XGB+LGBM OOF)
- E2: LightGBM single + sensor ablation
- E5: LightGBM + HistGBM native NaN (MIA leakage risk)
- E6: NaN ablation + SHAP leakage monitoring (THE publishable finding)

## Non-tree classifiers:
- E7: SVM (linear+RBF) + KNN + LogReg
- E7b: GaussianNB + LDA + QDA + Ridge + NearestCentroid

## Extended tree + imbalanced:
- E7c: HistGBM + CatBoost + ExtraTrees + BalancedRF

## Neural net:
- E7d: MLPClassifier (128-64, 256-128-64)

## Unsupervised:
- E8: Isolation Forest anomaly detection

## Summary:
- E9: Comparison table (all methods)
- E10: Save OOF + results JSON

## Parquets: bda_product_prepost (all experiments)

# CONFIG + GLOBAL SETUP

In [2]:
# @title CELL 3: NB09c CONFIG + GLOBAL SETUP
TIER_SELECTION = [0, 1,2]
CITY_SELECTION = None
REQUIRE_UNOSAT = True
RANDOM_STATE = 42

import platform, os
if platform.system() == "Windows":
    _setup = r"F:\PROJECTS\masterthesis\gdrive\masterthesis\notebooks\global_setup.py"
elif os.path.exists("/content/drive_f"):
    _setup = "/content/drive_f/masterthesis/notebooks/global_setup.py"
else:
    _setup = "/mnt/f/PROJECTS/masterthesis/gdrive/masterthesis/notebooks/global_setup.py"
with open(_setup) as f:
    exec(f.read())


BDA GLOBAL SETUP
Started: 2026-04-12 17:59:25
Python: 3.12.12

[1/7] Directory Structure
----------------------------------------------------------------------
  GDrive (G:):       /content/drive_f/masterthesis OK
  GDrive (F:):       /content/drive_f/masterthesis OK
  Local data (G:):   /content/masterthesis_local/data OK
  Data stack (F:):   /mnt/f/PROJECTS/masterthesis/data_stack OK

  TIER_SELECTION: [0, 1, 2]
  CITY_SELECTION: None (tier filter)
  REQUIRE_UNOSAT: True
  CITIES_TO_PROCESS: 21 cities

[2/7] Credentials
----------------------------------------------------------------------
  Copernicus: inf***
  OpenTopography: OK
  Earthdata: marcoheinzen

[3/7] Python Packages
----------------------------------------------------------------------

  Already installed: 23
  Newly installed:   0
  Failed:            0

[4/7] Global Imports & Configuration
----------------------------------------------------------------------
  All imports loaded

[5/7] Processing Config & SNAP
------

# CELL S0: PARQUET LOADER + SHARED EVALUATION

In [3]:
# @title CELL S0: NB09c PARQUET LOADER + SHARED EVALUATION
import sys, importlib, re, time, gc, json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              ExtraTreesClassifier, IsolationForest)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from sklearn.impute import SimpleImputer
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

print("=" * 70)
print("CELL S0: NB09c PARQUET LOADER")
print("=" * 70)

if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))
import stack_loader
importlib.reload(stack_loader)
from stack_loader import load_dataset, get_feature_groups

# ---- product_prepost (per-tier) ----
_tiers = TIER_SELECTION if isinstance(TIER_SELECTION, list) else [0, 1, 2]
df_pp = load_tier_parquets(PARQUET_PREPOST_TIER_FMT, _tiers)
df_bldg = load_tier_parquets(PARQUET_BUILDINGS_TIER_FMT, _tiers)

CITIES_TO_PROCESS, _battle_dates = resolve_cities(
    tier_selection=TIER_SELECTION,
    city_selection=[CITY_SELECTION] if isinstance(CITY_SELECTION, str) else CITY_SELECTION,
    require_unosat=REQUIRE_UNOSAT,
)

join_cols = ['building_id', 'city']
bldg_extra = [c for c in df_bldg.columns if c not in df_pp.columns]
df = df_pp.merge(df_bldg[join_cols + bldg_extra], on=join_cols, how='left')
del df_pp
df = df[df['city'].isin(CITIES_TO_PROCESS)].copy()
df = df[df['damage_binary'] >= 0].copy()
print(f"  product_prepost: {len(df)} buildings, {df['city'].nunique()} cities")

TARGET_COL = 'damage_binary'
N_FOLDS = 5
feature_groups = get_feature_groups(df)
EXCLUDE_GROUPS = {'meta', 'cloud_freq', 'obs_count', 'other'}

# feature lists
card_cols = [c for c in df.columns if re.match(r's1__(vv|vh)__(baseline|assessment)__', c)]
coh_cols = []
for gname, cols in feature_groups.items():
    if 'coh' in gname.lower() and gname not in EXCLUDE_GROUPS:
        coh_cols.extend([c for c in cols if c in df.columns])
coh_cols = list(dict.fromkeys(coh_cols))

ms_cols = []
for gname, cols in feature_groups.items():
    if gname not in EXCLUDE_GROUPS and any(gname.startswith(p) for p in ('comp_', 'lu_', 'landuse', 's2_', 'idx_', 'spectral', 'cd_', 'fire', 'vis')):
        ms_cols.extend([c for c in cols if c in df.columns])
ms_cols = list(dict.fromkeys(ms_cols))

all_feature_cols = []
for gname, cols in feature_groups.items():
    if gname not in EXCLUDE_GROUPS:
        all_feature_cols.extend([c for c in cols if c in df.columns])
all_feature_cols = list(dict.fromkeys(all_feature_cols))

print(f"  card: {len(card_cols)}, coh: {len(coh_cols)}, ms: {len(ms_cols)}, all: {len(all_feature_cols)}")

# ---- shared evaluation ----
EXPERIMENT_LOG = {}

def prepare_Xy(feat_cols, impute=True, add_was_observed=False):
    avail = [c for c in feat_cols if c in df.columns]
    if len(avail) < 2:
        return None, None, None, None
    df_valid = df[df[TARGET_COL] >= 0].copy()
    row_mask = ~df_valid[avail].isna().all(axis=1)
    df_sub = df_valid[row_mask]
    nan_col = df_sub[avail].isna().all()
    clean = [c for c in avail if not nan_col[c]]
    if len(clean) < 2:
        return None, None, None, None

    X = df_sub[clean].values.astype(np.float32)
    feat_names = list(clean)

    if add_was_observed:
        obs_flags = []
        obs_names = []
        for ci, col in enumerate(clean):
            nan_frac = np.isnan(X[:, ci]).mean()
            if nan_frac > 0.05:
                obs_flags.append((~np.isnan(X[:, ci])).astype(np.float32))
                obs_names.append(f'was_obs__{col}')
        if obs_flags:
            X = np.column_stack([X] + obs_flags)
            feat_names.extend(obs_names)

    if impute:
        imp = SimpleImputer(strategy='median')
        X = imp.fit_transform(X)

    y = df_sub[TARGET_COL].values
    groups = df_sub['city'].values
    return X, y, groups, feat_names

def evaluate_groupkfold(clf, X, y, groups, label="", n_folds=N_FOLDS, needs_scaling=False):
    n_cities = len(np.unique(groups))
    n_folds_actual = min(n_folds, n_cities)
    if n_folds_actual < 2:
        print(f"    SKIP {label}: only {n_cities} cities")
        return None
    gkf = GroupKFold(n_splits=n_folds_actual)
    y_proba_oof = np.full(len(y), np.nan)
    fold_aucs = []
    for train_idx, test_idx in gkf.split(X, y, groups):
        X_tr, X_te = X[train_idx], X[test_idx]
        if needs_scaling:
            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X_tr)
            X_te = scaler.transform(X_te)
        clf_copy = clone(clf)
        clf_copy.fit(X_tr, y[train_idx])
        if hasattr(clf_copy, 'predict_proba'):
            y_proba_oof[test_idx] = clf_copy.predict_proba(X_te)[:, 1]
        elif hasattr(clf_copy, 'decision_function'):
            raw = clf_copy.decision_function(X_te)
            y_proba_oof[test_idx] = 1.0 / (1.0 + np.exp(-raw))
        if len(np.unique(y[test_idx])) > 1:
            fold_aucs.append(roc_auc_score(y[test_idx], y_proba_oof[test_idx]))
    valid = ~np.isnan(y_proba_oof)
    auc = roc_auc_score(y[valid], y_proba_oof[valid])
    f1 = f1_score(y[valid], (y_proba_oof[valid] >= 0.5).astype(int))
    res = {'auc': auc, 'f1': f1, 'auc_std': np.std(fold_aucs), 'fold_aucs': fold_aucs,
           'y_true': y[valid], 'y_proba': y_proba_oof[valid], 'groups': groups[valid],
           'n_features': X.shape[1], 'experiment': label}
    EXPERIMENT_LOG[label] = res
    print(f"    {label:45s}: AUC={auc:.3f} (+/-{np.std(fold_aucs):.3f})  F1={f1:.3f}")
    return res

print(f"  Functions: prepare_Xy(impute, add_was_observed), evaluate_groupkfold(needs_scaling)")




# ---- RESULT REGISTRY (for NB13 consolidation) ----
from bda_results import ResultRegistry
registry = ResultRegistry(RESULTS_ROOT, notebook='NB09c')

def log_result(res, cell_id='', parquet_name='bda_product_prepost', feature_set_name='',
               classifier_name='', feature_cols=None, groups=None,
               cv_method='GroupKFold', imputation='median', note='', tags=None):
    if res is None:
        return
    registry.log_experiment(
        cell_id=cell_id,
        experiment_name=res.get('experiment', ''),
        parquet_name=parquet_name,
        tier_selection=TIER_SELECTION if isinstance(TIER_SELECTION, list) else [0,1,2],
        classifier_name=classifier_name,
        feature_set_name=feature_set_name,
        feature_cols=feature_cols,
        cv_method=cv_method, n_folds=N_FOLDS, imputation=imputation,
        y_true=res.get('y_true'), y_proba=res.get('y_proba'),
        groups=res.get('groups', groups),
        note=note, tags=tags or [],
    )

# ---- OUTPUT + SAVE HELPERS ----
import matplotlib.pyplot as plt
from datetime import datetime as _dt

OUT_DIR = RESULTS_ROOT / 'nb09c'
OUT_DIR.mkdir(parents=True, exist_ok=True)

def save_result(data, name, cell_id, fmt='csv'):
    cell_dir = OUT_DIR / cell_id
    cell_dir.mkdir(parents=True, exist_ok=True)
    ts = _dt.now().strftime('%Y%m%d_%H%M%S')
    if fmt == 'csv' and isinstance(data, pd.DataFrame):
        path = cell_dir / f"{name}_{ts}.csv"
        data.to_csv(path, index=False)
    elif fmt == 'json':
        path = cell_dir / f"{name}_{ts}.json"
        import json as _j
        with open(path, 'w') as fh:
            _j.dump(data, fh, indent=2, default=str)
    else:
        raise ValueError(f"Unknown fmt={fmt}")
    print(f"  Saved: {path.relative_to(OUT_DIR)} ({path.stat().st_size / 1024:.1f} KB)")
    return path

def save_fig(fig, name, cell_id, dpi=150):
    cell_dir = OUT_DIR / cell_id
    cell_dir.mkdir(parents=True, exist_ok=True)
    ts = _dt.now().strftime('%Y%m%d_%H%M%S')
    path = cell_dir / f"{name}_{ts}.png"
    fig.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"  Plot: {path.relative_to(OUT_DIR)}")
    return path

print(f"  Output: {OUT_DIR}")
print(f"  Helpers: save_result(), save_fig()")


CELL S0: NB09c PARQUET LOADER
  load_tier_parquets: 3 tiers, 907371 rows
  load_tier_parquets: 3 tiers, 907371 rows
  product_prepost: 598595 buildings, 21 cities
  card: 96, coh: 18, ms: 192, all: 311
  Functions: prepare_Xy(impute, add_was_observed), evaluate_groupkfold(needs_scaling)
  ResultRegistry: /content/drive_f/masterthesis/results/registry (run_id=20260412_180131)
  Output: /content/drive_f/masterthesis/results/nb09c
  Helpers: save_result(), save_fig()


# CELL E1: STACKING META-LEARNER

In [4]:
# @title CELL E1: STACKING META-LEARNER (LogReg on base model OOF)
# =============================================================================
# Train RF + GBM + XGB independently with GroupKFold.
# Use OOF predictions as features for LogReg meta-learner.
# Also: soft-voting baseline for comparison.
# Parquet: bda_product_prepost (all features).
# =============================================================================
print("=" * 70)
print("CELL E1: STACKING META-LEARNER")
print("=" * 70)

X, y, groups, feat_names = prepare_Xy(all_feature_cols)
if X is None:
    print("  Insufficient features")
else:
    n_cities = len(np.unique(groups))
    n_folds = min(N_FOLDS, n_cities)
    gkf = GroupKFold(n_splits=n_folds)

    base_models = {
        'RF': RandomForestClassifier(n_estimators=200, min_samples_leaf=3, random_state=RANDOM_STATE,
                                      n_jobs=-1, class_weight='balanced'),
        'GBM': GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=5,
                                           random_state=RANDOM_STATE),
    }
    if HAS_XGB:
        base_models['XGB'] = XGBClassifier(n_estimators=200, random_state=RANDOM_STATE,
                                            scale_pos_weight=5, eval_metric='logloss', verbosity=0)
    if HAS_LGBM:
        base_models['LGBM'] = LGBMClassifier(n_estimators=200, random_state=RANDOM_STATE,
                                              class_weight='balanced', verbose=-1)

    # generate OOF predictions per base model
    oof_dict = {}
    print(f"\n  Base model OOF predictions ({n_folds}-fold GroupKFold):")
    for name, clf in base_models.items():
        oof = np.full(len(y), np.nan)
        for train_idx, test_idx in gkf.split(X, y, groups):
            clf_copy = clone(clf)
            clf_copy.fit(X[train_idx], y[train_idx])
            oof[test_idx] = clf_copy.predict_proba(X[test_idx])[:, 1]
        valid = ~np.isnan(oof)
        auc = roc_auc_score(y[valid], oof[valid])
        oof_dict[name] = oof
        print(f"    {name:10s}: AUC={auc:.3f}")
        EXPERIMENT_LOG[f'E1_base_{name}'] = {'auc': auc, 'f1': f1_score(y[valid], (oof[valid]>=0.5).astype(int)),
                                              'n_features': X.shape[1]}
        registry.log_experiment(
            cell_id='cell_e1', experiment_name=f'E1_base_{name}',
            parquet_name='bda_product_prepost', feature_set_name='all_multimodal',
            classifier_name=name, feature_cols=feat_names,
            cv_method='GroupKFold', n_folds=n_folds, imputation='median',
            y_true=y[valid], y_proba=oof[valid], groups=groups[valid],
            note=f'E1 stacking base model: {name}', tags=['stacking', 'base_model', name],
        )

    # stacking: LogReg on OOF predictions
    oof_matrix = np.column_stack([oof_dict[n] for n in oof_dict])
    valid_all = ~np.any(np.isnan(oof_matrix), axis=1)
    X_stack = oof_matrix[valid_all]
    y_stack = y[valid_all]
    groups_stack = groups[valid_all]

    meta = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
    res_stack = evaluate_groupkfold(meta, X_stack, y_stack, groups_stack, "E1_stacking_LogReg", needs_scaling=True)
    log_result(res_stack, cell_id='cell_e1', feature_set_name='oof_predictions',
               classifier_name='LogReg_meta', feature_cols=list(base_models.keys()),
               note='Stacking meta-learner: LogReg on base model OOF predictions',
               tags=['stacking', 'meta_learner'])

    # soft voting baseline
    avg_proba = np.nanmean(oof_matrix, axis=1)
    valid_avg = ~np.isnan(avg_proba)
    auc_vote = roc_auc_score(y[valid_avg], avg_proba[valid_avg])
    f1_vote = f1_score(y[valid_avg], (avg_proba[valid_avg] >= 0.5).astype(int))
    EXPERIMENT_LOG['E1_soft_voting'] = {'auc': auc_vote, 'f1': f1_vote, 'n_features': len(base_models)}
    print(f"    {'E1_soft_voting':45s}: AUC={auc_vote:.3f}  F1={f1_vote:.3f}")
    registry.log_experiment(
        cell_id='cell_e1', experiment_name='E1_soft_voting',
        parquet_name='bda_product_prepost', feature_set_name='all_multimodal',
        classifier_name='SoftVoting', feature_cols=feat_names,
        cv_method='GroupKFold', n_folds=n_folds, imputation='median',
        y_true=y[valid_avg], y_proba=avg_proba[valid_avg], groups=groups[valid_avg],
        note='Soft voting: average of base model OOF probabilities',
        tags=['stacking', 'soft_voting'],
    )



CELL E1: STACKING META-LEARNER

  Base model OOF predictions (5-fold GroupKFold):
    RF        : AUC=0.421
  REG: E1_base_RF                                    AUC=0.4210 F1=0.0019 n=598595 feat=311 cities=21 [NB09c/cell_e1]
    GBM       : AUC=0.658
  REG: E1_base_GBM                                   AUC=0.6577 F1=0.0611 n=598595 feat=311 cities=21 [NB09c/cell_e1]
    XGB       : AUC=0.544
  REG: E1_base_XGB                                   AUC=0.5441 F1=0.0355 n=598595 feat=311 cities=21 [NB09c/cell_e1]
    LGBM      : AUC=0.578
  REG: E1_base_LGBM                                  AUC=0.5781 F1=0.0543 n=598595 feat=311 cities=21 [NB09c/cell_e1]
    E1_stacking_LogReg                           : AUC=0.494 (+/-0.050)  F1=0.000
  REG: E1_stacking_LogReg                            AUC=0.4938 F1=0.0000 n=598595 feat=4 cities=21 [NB09c/cell_e1]
    E1_soft_voting                               : AUC=0.571  F1=0.028
  REG: E1_soft_voting                                AUC=0.5712 F1=0.0277

# CELL E2: LightGBM

In [5]:
# @title CELL E2: LightGBM (best single classifier candidate)
print("=" * 70)
print("CELL E2: LightGBM SINGLE MODEL")
print("=" * 70)

if not HAS_LGBM:
    print("  LightGBM not installed")
else:
    X, y, groups, feat_names = prepare_Xy(all_feature_cols)
    if X is not None:
        lgbm = LGBMClassifier(n_estimators=200, random_state=RANDOM_STATE, class_weight='balanced', verbose=-1)
        res = evaluate_groupkfold(lgbm, X, y, groups, "E2_LightGBM_all")
        log_result(res, cell_id='cell_e2', feature_set_name='all_multimodal',
                   classifier_name='LightGBM', feature_cols=feat_names,
                   note='LightGBM single model on all features',
                   tags=['lightgbm', 'single_model'])

        # sensor ablation
        for gname, gcols in [('card', card_cols), ('coh', coh_cols), ('ms', ms_cols),
                              ('card+coh', list(dict.fromkeys(card_cols + coh_cols)))]:
            X_g, y_g, groups_g, _ = prepare_Xy(gcols)
            if X_g is not None:
                res_g = evaluate_groupkfold(lgbm, X_g, y_g, groups_g, f"E2_LightGBM_{gname}")
                log_result(res_g, cell_id='cell_e2', feature_set_name=gname,
                           classifier_name='LightGBM',
                           note=f'LightGBM sensor ablation: {gname}',
                           tags=['lightgbm', 'sensor_ablation', gname])



CELL E2: LightGBM SINGLE MODEL
    E2_LightGBM_all                              : AUC=0.578 (+/-0.073)  F1=0.054
  REG: E2_LightGBM_all                               AUC=0.5781 F1=0.0543 n=598595 feat=311 cities=21 [NB09c/cell_e2]
    E2_LightGBM_card                             : AUC=0.606 (+/-0.094)  F1=0.059
  REG: E2_LightGBM_card                              AUC=0.6065 F1=0.0588 n=368288 feat=0 cities=21 [NB09c/cell_e2]
    E2_LightGBM_coh                              : AUC=0.599 (+/-0.063)  F1=0.173
  REG: E2_LightGBM_coh                               AUC=0.5986 F1=0.1735 n=79138 feat=0 cities=9 [NB09c/cell_e2]
    E2_LightGBM_ms                               : AUC=0.638 (+/-0.027)  F1=0.068
  REG: E2_LightGBM_ms                                AUC=0.6385 F1=0.0680 n=310131 feat=0 cities=19 [NB09c/cell_e2]
    E2_LightGBM_card+coh                         : AUC=0.614 (+/-0.112)  F1=0.078
  REG: E2_LightGBM_card+coh                          AUC=0.6145 F1=0.0781 n=368288 feat=0 citie

# CELL E5: LightGBM NATIVE NaN (MIA — leakage risk)

In [6]:
# @title CELL E5: LightGBM NATIVE NaN (NO IMPUTATION)
# =============================================================================
# LightGBM/XGBoost handle NaN natively (MIA). DANGEROUS when NaN encodes
# site identity (cloud cover patterns differ by city = geographic proxy).
# Parquet: bda_product_prepost (all features, NO imputation).
# =============================================================================
print("=" * 70)
print("CELL E5: LightGBM NATIVE NaN")
print("=" * 70)

if not HAS_LGBM:
    print("  LightGBM not installed")
else:
    # NO imputation — NaN stays
    X_raw, y_raw, groups_raw, feat_raw = prepare_Xy(all_feature_cols, impute=False)
    if X_raw is not None:
        nan_pct = np.isnan(X_raw).mean() * 100
        print(f"  Features: {X_raw.shape[1]}, NaN: {nan_pct:.1f}%")
        for city in np.unique(groups_raw):
            mask = groups_raw == city
            pct = np.isnan(X_raw[mask]).mean() * 100
            print(f"    {city:25s}: {pct:.1f}% NaN")

        lgbm_nan = LGBMClassifier(n_estimators=200, random_state=RANDOM_STATE,
                                   class_weight='balanced', verbose=-1)
        res = evaluate_groupkfold(lgbm_nan, X_raw, y_raw, groups_raw, "E5_LightGBM_native_NaN")
        log_result(res, cell_id='cell_e5', feature_set_name='all_multimodal',
                   classifier_name='LightGBM_nativeNaN', imputation='none',
                   note='LightGBM native NaN (MIA) — leakage risk if NaN encodes city',
                   tags=['lightgbm', 'native_nan', 'leakage_risk'])

        # compare with imputed version
        imputed_auc = EXPERIMENT_LOG.get('E2_LightGBM_all', {}).get('auc', np.nan)
        native_auc = EXPERIMENT_LOG.get('E5_LightGBM_native_NaN', {}).get('auc', np.nan)
        if not np.isnan(imputed_auc) and not np.isnan(native_auc):
            delta = native_auc - imputed_auc
            print(f"\n  Native NaN vs median impute: {delta:+.3f} AUC")
            print(f"  {'LEAKAGE RISK' if delta > 0.02 else 'Safe'}: native NaN {'outperforms' if delta > 0 else 'underperforms'} imputed")

    # HistGradientBoostingClassifier — sklearn native NaN handler
    from sklearn.ensemble import HistGradientBoostingClassifier
    hgbm = HistGradientBoostingClassifier(max_iter=200, random_state=RANDOM_STATE,
                                           class_weight='balanced', early_stopping=True)
    res_hgbm = evaluate_groupkfold(hgbm, X_raw, y_raw, groups_raw, "E5_HistGBM_native_NaN")
    log_result(res_hgbm, cell_id='cell_e5', feature_set_name='all_multimodal',
               classifier_name='HistGBM_nativeNaN', imputation='none',
               note='HistGradientBoosting native NaN',
               tags=['histgbm', 'native_nan'])

    # compare all native NaN methods
    print(f"\n  Native NaN comparison:")
    for key in ['E2_LightGBM_all', 'E5_LightGBM_native_NaN', 'E5_HistGBM_native_NaN']:
        auc = EXPERIMENT_LOG.get(key, {}).get('auc', np.nan)
        if not np.isnan(auc):
            print(f"    {key:40s}: AUC={auc:.3f}")


CELL E5: LightGBM NATIVE NaN
  Features: 311, NaN: 56.7%
    Avdiivka                 : 49.3% NaN
    Borodyanka               : 84.8% NaN
    Bucha                    : 41.3% NaN
    Chernihiv                : 53.3% NaN
    Chornobaivka             : 25.8% NaN
    Dmytrivka                : 52.0% NaN
    Hostomel                 : 44.6% NaN
    Irpin                    : 56.5% NaN
    Kharkiv                  : 47.8% NaN
    Kherson                  : 25.8% NaN
    Kramatorsk               : 44.3% NaN
    Lysychansk               : 50.9% NaN
    Makariv                  : 60.6% NaN
    Mariupol                 : 43.2% NaN
    Moschun                  : 60.4% NaN
    Mykolaiv                 : 90.9% NaN
    Okhtyrka                 : 55.4% NaN
    Rubizhne                 : 51.6% NaN
    Sievierodonetsk          : 35.3% NaN
    Trostianets              : 63.2% NaN
    Volnovakha               : 50.4% NaN
    E5_LightGBM_native_NaN                       : AUC=0.514 (+/-0.090)  F1=0.041


# CELL E6: NaN ABLATION + SHAP LEAKAGE MONITORING

THE publishable finding: NaN strategy > classifier choice.

In [7]:
# @title CELL E6: NaN ABLATION + SHAP LEAKAGE MONITORING
# =============================================================================
# Three strategies x modality groups:
#   A: Global median impute + was_observed binary flags
#   B: LightGBM native NaN (MIA)
#   C: SAR-only (zero MS NaN — tests if NaN from cloud masking is the issue)
#
# SHAP: if was_observed flags appear in top-10, model exploits availability as proxy.
# Prior finding: SAR coh_only AUC 0.47->0.74 by keeping all rows.
# =============================================================================
print("=" * 70)
print("CELL E6: NaN ABLATION + SHAP LEAKAGE")
print("=" * 70)

modality_configs = {
    'card_only': card_cols,
    'coh_only': coh_cols,
    'card+coh': list(dict.fromkeys(card_cols + coh_cols)),
    'all_multimodal': all_feature_cols,
}

results_nan = {}

# ---- STRATEGY A: Median impute + was_observed flags ----
print(f"\n  STRATEGY A: Global median impute + was_observed flags")
for mname, mcols in modality_configs.items():
    X, y, groups, fnames = prepare_Xy(mcols, impute=True, add_was_observed=True)
    if X is None:
        continue
    n_obs_flags = sum(1 for f in fnames if f.startswith('was_obs__'))
    rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=3, random_state=RANDOM_STATE,
                                 n_jobs=-1, class_weight='balanced')
    res = evaluate_groupkfold(rf, X, y, groups, f"E6_A_impute+obs_{mname}")
    if res:
        results_nan[f'A_{mname}'] = {**res, 'n_obs_flags': n_obs_flags, 'feat_names': fnames}
        log_result(res, cell_id='cell_e6', feature_set_name=f'A_{mname}',
                   classifier_name='RF-200', feature_cols=fnames,
                   note=f'NaN ablation Strategy A: median impute + was_observed, {mname}',
                   tags=['nan_ablation', 'strategy_a', mname])

# ---- STRATEGY B: LightGBM native NaN ----
if HAS_LGBM:
    print(f"\n  STRATEGY B: LightGBM native NaN (MIA)")
    for mname, mcols in modality_configs.items():
        X, y, groups, fnames = prepare_Xy(mcols, impute=False)
        if X is None:
            continue
        lgbm = LGBMClassifier(n_estimators=200, random_state=RANDOM_STATE,
                               class_weight='balanced', verbose=-1)
        res = evaluate_groupkfold(lgbm, X, y, groups, f"E6_B_nativeNaN_{mname}")
        if res:
            results_nan[f'B_{mname}'] = res
            log_result(res, cell_id='cell_e6', feature_set_name=f'B_{mname}',
                       classifier_name='LightGBM_nativeNaN', imputation='none',
                       note=f'NaN ablation Strategy B: LightGBM native NaN, {mname}',
                       tags=['nan_ablation', 'strategy_b', mname])

# ---- STRATEGY C: SAR-only (zero MS NaN) ----
print(f"\n  STRATEGY C: SAR-only subset (no MS -> no cloud NaN)")
sar_cols = list(dict.fromkeys(card_cols + coh_cols))
X_sar, y_sar, groups_sar, fnames_sar = prepare_Xy(sar_cols, impute=True)
if X_sar is not None:
    rf_sar = RandomForestClassifier(n_estimators=200, min_samples_leaf=3, random_state=RANDOM_STATE,
                                     n_jobs=-1, class_weight='balanced')
    res_sar = evaluate_groupkfold(rf_sar, X_sar, y_sar, groups_sar, "E6_C_SAR_only_no_NaN")
    log_result(res_sar, cell_id='cell_e6', feature_set_name='C_sar_only',
               classifier_name='RF-200', feature_cols=fnames_sar,
               note='NaN ablation Strategy C: SAR-only (zero MS NaN)',
               tags=['nan_ablation', 'strategy_c', 'sar_only'])

# ---- SHAP leakage check (on Strategy A all_multimodal) ----
try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False

if HAS_SHAP and 'A_all_multimodal' in results_nan:
    print(f"\n  SHAP leakage check (Strategy A, all_multimodal):")
    X_shap, y_shap, groups_shap, fnames_shap = prepare_Xy(all_feature_cols, impute=True, add_was_observed=True)
    if X_shap is not None:
        rf_shap = RandomForestClassifier(n_estimators=100, min_samples_leaf=3, random_state=RANDOM_STATE,
                                          n_jobs=-1, class_weight='balanced')
        rf_shap.fit(X_shap, y_shap)
        explainer = shap.TreeExplainer(rf_shap)
        shap_values = explainer.shap_values(X_shap[:500], check_additivity=False)
        if isinstance(shap_values, list):
            shap_values = shap_values[1]
        mean_abs_shap = np.abs(shap_values).mean(axis=0)
        top_idx = np.argsort(mean_abs_shap)[-15:][::-1]

        print(f"    Top-15 SHAP features:")
        obs_in_top10 = 0
        for rank, idx in enumerate(top_idx):
            for rank, idx in enumerate(top_idx):
                idx = int(idx)
                fname = fnames_shap[idx] if idx < len(fnames_shap) else f"feat_{idx}"
            is_obs = fname.startswith('was_obs__')
            tag = " *** LEAKAGE INDICATOR" if is_obs else ""
            print(f"      {rank+1:2d}. {fname:50s} SHAP={mean_abs_shap[idx]:.4f}{tag}")
            if is_obs and rank < 10:
                obs_in_top10 += 1

        if obs_in_top10 > 0:
            print(f"\n    WARNING: {obs_in_top10} was_observed flags in SHAP top-10!")
            print(f"    Model exploits modality availability as geographic proxy.")
            print(f"    NaN patterns encode city identity -> leakage vector.")
        else:
            print(f"\n    OK: No was_observed flags in SHAP top-10. NaN patterns are safe.")
else:
    print(f"\n  SHAP: {'not installed' if not HAS_SHAP else 'no Strategy A results'}")

# ---- Summary table ----
print(f"\n  {'Strategy':<45s} {'AUC':>7s}")
print(f"  {'-'*45} {'-'*7}")
for key in sorted(results_nan.keys()):
    auc = results_nan[key].get('auc', np.nan)
    print(f"  {key:<45s} {auc:>7.3f}")



CELL E6: NaN ABLATION + SHAP LEAKAGE

  STRATEGY A: Global median impute + was_observed flags
    E6_A_impute+obs_card_only                    : AUC=0.448 (+/-0.185)  F1=0.000
  REG: E6_A_impute+obs_card_only                     AUC=0.4475 F1=0.0000 n=368288 feat=138 cities=21 [NB09c/cell_e6]
    E6_A_impute+obs_coh_only                     : AUC=0.545 (+/-0.056)  F1=0.031
  REG: E6_A_impute+obs_coh_only                      AUC=0.5451 F1=0.0312 n=79138 feat=18 cities=9 [NB09c/cell_e6]
    E6_A_impute+obs_card+coh                     : AUC=0.502 (+/-0.109)  F1=0.015
  REG: E6_A_impute+obs_card+coh                      AUC=0.5018 F1=0.0149 n=368288 feat=174 cities=21 [NB09c/cell_e6]
    E6_A_impute+obs_all_multimodal               : AUC=0.515 (+/-0.049)  F1=0.008
  REG: E6_A_impute+obs_all_multimodal                AUC=0.5151 F1=0.0079 n=598595 feat=620 cities=21 [NB09c/cell_e6]

  STRATEGY B: LightGBM native NaN (MIA)
    E6_B_nativeNaN_card_only                     : AUC=0.605 (+/-0.0

TypeError: only length-1 arrays can be converted to Python scalars

# CELL E7: NON-TREE CLASSIFIERS (SVM + KNN)

Non-tree models may behave differently with NaN patterns and correlated features. SVM needs scaling. KNN captures spatial neighborhoods.

In [ ]:
# @title CELL E7: SVM + KNN (non-tree baselines)
# =============================================================================
# Non-tree classifiers that may behave differently with NaN patterns:
# - SVM (linear): sparse-data-friendly, different decision boundary
# - SVM (RBF): captures nonlinear patterns
# - KNN: neighborhood-based, spatial autocorrelation reference
# All require scaling. All use median imputation (no native NaN support).
# Parquet: bda_product_prepost.
# =============================================================================
print("=" * 70)
print("CELL E7: NON-TREE CLASSIFIERS")
print("=" * 70)

X, y, groups, feat_names = prepare_Xy(all_feature_cols)
if X is None:
    print("  Insufficient features")
else:
    classifiers = {
        'SVM_linear': SVC(kernel='linear', probability=True, class_weight='balanced',
                          random_state=RANDOM_STATE, max_iter=5000),
        'SVM_rbf': SVC(kernel='rbf', probability=True, class_weight='balanced',
                        random_state=RANDOM_STATE, max_iter=5000),
        'KNN_5': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        'KNN_15': KNeighborsClassifier(n_neighbors=15, n_jobs=-1),
        'LogReg': LogisticRegression(class_weight='balanced', random_state=RANDOM_STATE, max_iter=1000),
    }

    for name, clf in classifiers.items():
        res = evaluate_groupkfold(clf, X, y, groups, f"E7_{name}", needs_scaling=True)
        log_result(res, cell_id='cell_e7', feature_set_name='all_multimodal',
                   classifier_name=name, feature_cols=feat_names,
                   note=f'Non-tree classifier: {name}',
                   tags=['non_tree', name])



# CELL E7b: PROBABILISTIC + DISCRIMINANT CLASSIFIERS

GaussianNB (feature-independent), LDA/QDA (discriminant analysis), RidgeClassifier (linear diagnostic), NearestCentroid (trivial baseline). All bring fundamentally different inductive biases from tree ensembles.

In [ ]:
# @title CELL E7b: PROBABILISTIC + DISCRIMINANT CLASSIFIERS
# =============================================================================
# Methods with different inductive biases than trees:
# - GaussianNB: assumes feature independence (diagnostic — if close to trees,
#   features individually carry the signal, not interactions)
# - LDA: shared covariance, projects to most discriminant axis (also dimensionality reduction)
# - QDA: per-class covariance (captures if damage changes VARIANCE not just mean)
# - RidgeClassifier: L2-regularized linear (if good, signal is linearly separable)
# - NearestCentroid: classify by class mean proximity (trivial sanity check)
# All require scaling. Parquet: bda_product_prepost.
# =============================================================================
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import RidgeClassifier
from sklearn.neighbors import NearestCentroid

print("=" * 70)
print("CELL E7b: PROBABILISTIC + DISCRIMINANT CLASSIFIERS")
print("=" * 70)

X, y, groups, feat_names = prepare_Xy(all_feature_cols)
if X is None:
    print("  Insufficient features")
else:
    # GaussianNB — no scaling needed, handles its own normalization
    gnb = GaussianNB()
    res_gaussiannb = evaluate_groupkfold(gnb, X, y, groups, "E7b_GaussianNB")
    log_result(res_gaussiannb, cell_id='cell_e7b', feature_set_name='all_multimodal',
               classifier_name='GaussianNB', feature_cols=feat_names,
               note='Probabilistic/discriminant classifier: GaussianNB',
               tags=['probabilistic', 'gaussiannb'])

    # LDA — shared covariance, shrinkage for stability with many features
    lda = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
    res_lda = evaluate_groupkfold(lda, X, y, groups, "E7b_LDA", needs_scaling=True)
    log_result(res_lda, cell_id='cell_e7b', feature_set_name='all_multimodal',
               classifier_name='LDA', feature_cols=feat_names,
               note='Probabilistic/discriminant classifier: LDA',
               tags=['probabilistic', 'lda'])

    # QDA — per-class covariance, needs strong regularization with 300 features
    qda = QuadraticDiscriminantAnalysis(reg_param=0.3)
    res_qda = evaluate_groupkfold(qda, X, y, groups, "E7b_QDA", needs_scaling=True)
    log_result(res_qda, cell_id='cell_e7b', feature_set_name='all_multimodal',
               classifier_name='QDA', feature_cols=feat_names,
               note='Probabilistic/discriminant classifier: QDA',
               tags=['probabilistic', 'qda'])

    # RidgeClassifier — L2-regularized linear
    ridge = RidgeClassifier(alpha=1.0, class_weight='balanced')
    res_ridgeclassifier = evaluate_groupkfold(ridge, X, y, groups, "E7b_RidgeClassifier", needs_scaling=True)
    log_result(res_ridgeclassifier, cell_id='cell_e7b', feature_set_name='all_multimodal',
               classifier_name='RidgeClassifier', feature_cols=feat_names,
               note='Probabilistic/discriminant classifier: RidgeClassifier',
               tags=['probabilistic', 'ridgeclassifier'])

    # NearestCentroid — trivial prototype baseline
    nc = NearestCentroid()
    res_nearestcentroid = evaluate_groupkfold(nc, X, y, groups, "E7b_NearestCentroid", needs_scaling=True)
    log_result(res_nearestcentroid, cell_id='cell_e7b', feature_set_name='all_multimodal',
               classifier_name='NearestCentroid', feature_cols=feat_names,
               note='Probabilistic/discriminant classifier: NearestCentroid',
               tags=['probabilistic', 'nearestcentroid'])

    # interpretation
    gnb_auc = EXPERIMENT_LOG.get('E7b_GaussianNB', {}).get('auc', np.nan)
    lgbm_auc = EXPERIMENT_LOG.get('E2_LightGBM_all', {}).get('auc', np.nan)
    if not np.isnan(gnb_auc) and not np.isnan(lgbm_auc):
        gap = lgbm_auc - gnb_auc
        if gap < 0.03:
            print(f"\n  GaussianNB within {gap:.3f} of LightGBM -> features individually discriminative")
        else:
            print(f"\n  GaussianNB {gap:.3f} below LightGBM -> feature interactions matter")



# CELL E7c: EXTENDED TREE ENSEMBLE + IMBALANCED LEARNING

HistGBM (sklearn native NaN), CatBoost (ordered boosting, ESA WorldCover validated), ExtraTrees (random splits), BalancedRandomForest (sampling-based class balancing).

In [ ]:
# @title CELL E7c: EXTENDED TREE ENSEMBLE + IMBALANCED LEARNING
# =============================================================================
# Tree variants with genuinely different mechanisms:
# - HistGBM: sklearn native, histogram binning, native NaN (here with imputation for fair comparison)
# - CatBoost: ordered boosting (reduces target leakage within trees), ESA WorldCover production model
# - ExtraTrees: random split thresholds (higher bias, lower variance than RF)
# - BalancedRandomForest: random undersampling per bootstrap (imblearn)
# Parquet: bda_product_prepost.
# =============================================================================
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier

print("=" * 70)
print("CELL E7c: EXTENDED TREE ENSEMBLE + IMBALANCED")
print("=" * 70)

X, y, groups, feat_names = prepare_Xy(all_feature_cols)
if X is None:
    print("  Insufficient features")
else:
    # HistGradientBoostingClassifier (with imputation for fair comparison vs E5 native NaN)
    hgbm = HistGradientBoostingClassifier(max_iter=200, random_state=RANDOM_STATE,
                                           class_weight='balanced', early_stopping=True)
    res_histgbm = evaluate_groupkfold(hgbm, X, y, groups, "E7c_HistGBM_imputed")
        log_result(res_histgbm, cell_id='cell_e7c', feature_set_name='all_multimodal',
                   classifier_name='HistGBM', feature_cols=feat_names,
                   note='Extended tree ensemble: HistGBM',
                   tags=['tree_ensemble', 'histgbm'])

    # ExtraTreesClassifier
    et = ExtraTreesClassifier(n_estimators=200, min_samples_leaf=3, random_state=RANDOM_STATE,
                               n_jobs=-1, class_weight='balanced')
    res_extratrees = evaluate_groupkfold(et, X, y, groups, "E7c_ExtraTrees")
        log_result(res_extratrees, cell_id='cell_e7c', feature_set_name='all_multimodal',
                   classifier_name='ExtraTrees', feature_cols=feat_names,
                   note='Extended tree ensemble: ExtraTrees',
                   tags=['tree_ensemble', 'extratrees'])

    # CatBoost
    try:
        from catboost import CatBoostClassifier
        cat = CatBoostClassifier(iterations=200, random_seed=RANDOM_STATE, auto_class_weights='Balanced',
                                  verbose=0, allow_writing_files=False)
        res_catboost = evaluate_groupkfold(cat, X, y, groups, "E7c_CatBoost")
        log_result(res_catboost, cell_id='cell_e7c', feature_set_name='all_multimodal',
                   classifier_name='CatBoost', feature_cols=feat_names,
                   note='Extended tree ensemble: CatBoost',
                   tags=['tree_ensemble', 'catboost'])
    except ImportError:
        print("    CatBoost not installed, skipping")

    # BalancedRandomForestClassifier (imblearn)
    try:
        from imblearn.ensemble import BalancedRandomForestClassifier
        brf = BalancedRandomForestClassifier(n_estimators=200, min_samples_leaf=3, random_state=RANDOM_STATE,
                                              n_jobs=-1)
        res_balancedrf = evaluate_groupkfold(brf, X, y, groups, "E7c_BalancedRF")
        log_result(res_balancedrf, cell_id='cell_e7c', feature_set_name='all_multimodal',
                   classifier_name='BalancedRF', feature_cols=feat_names,
                   note='Extended tree ensemble: BalancedRF',
                   tags=['tree_ensemble', 'balancedrf'])

        # compare sampling vs cost-weighting
        rf_cost_auc = EXPERIMENT_LOG.get('E1_base_RF', {}).get('auc', np.nan)
        brf_auc = EXPERIMENT_LOG.get('E7c_BalancedRF', {}).get('auc', np.nan)
        if not np.isnan(rf_cost_auc) and not np.isnan(brf_auc):
            print(f"\n  Sampling vs cost-weighting: BalancedRF={brf_auc:.3f} vs RF(class_weight)={rf_cost_auc:.3f}")
    except ImportError:
        print("    imblearn not installed, skipping BalancedRF")



# CELL E7d: MLP NEURAL NETWORK BASELINE

Smooth nonlinear boundaries via weighted feature combinations. Grinsztajn et al. 2022 (NeurIPS) found trees beat MLP on tabular data — verify this holds for BDA features.

In [ ]:
# @title CELL E7d: MLP NEURAL NETWORK BASELINE
# =============================================================================
# MLPClassifier learns smooth nonlinear boundaries — fundamentally different from
# axis-aligned tree splits. Known to underperform trees on tabular data
# (Grinsztajn et al. 2022 NeurIPS), but our RS features are all numerical and
# relatively smooth, which partially mitigates MLP weaknesses.
# Requires StandardScaler. Parquet: bda_product_prepost.
# =============================================================================
from sklearn.neural_network import MLPClassifier

print("=" * 70)
print("CELL E7d: MLP NEURAL NETWORK BASELINE")
print("=" * 70)

X, y, groups, feat_names = prepare_Xy(all_feature_cols)
if X is None:
    print("  Insufficient features")
else:
    configs = {
        'MLP_128_64': MLPClassifier(hidden_layer_sizes=(128, 64), activation='relu', solver='adam',
                                     alpha=1e-4, early_stopping=True, random_state=RANDOM_STATE, max_iter=500),
        'MLP_256_128_64': MLPClassifier(hidden_layer_sizes=(256, 128, 64), activation='relu', solver='adam',
                                         alpha=1e-4, early_stopping=True, random_state=RANDOM_STATE, max_iter=500),
    }

    for name, clf in configs.items():
        res = evaluate_groupkfold(clf, X, y, groups, f"E7d_{name}", needs_scaling=True)
        log_result(res, cell_id='cell_e7d', feature_set_name='all_multimodal',
                   classifier_name=name, feature_cols=feat_names,
                   note=f'MLP neural network: {name}',
                   tags=['mlp', 'neural_network', name])

    # compare with LightGBM
    lgbm_auc = EXPERIMENT_LOG.get('E2_LightGBM_all', {}).get('auc', np.nan)
    best_mlp = max((EXPERIMENT_LOG.get(f'E7d_{n}', {}).get('auc', 0) for n in configs), default=0)
    if not np.isnan(lgbm_auc) and best_mlp > 0:
        print(f"\n  MLP best={best_mlp:.3f} vs LightGBM={lgbm_auc:.3f} (delta={best_mlp-lgbm_auc:+.3f})")
        print(f"  Grinsztajn 2022 prediction: trees > MLP on tabular. {'Confirmed' if lgbm_auc > best_mlp else 'VIOLATED'}.")



# CELL E8: ISOLATION FOREST (anomaly detection)

Unsupervised: treats damaged buildings as anomalies. Different from NB09b threshold methods which use single features. IF uses the full feature space.

In [ ]:
# @title CELL E8: ISOLATION FOREST (anomaly detection, unsupervised)
# =============================================================================
# Buildings with unusual feature patterns = potential damage.
# No training labels needed — evaluate against UNOSAT labels post-hoc.
# Different from NB09b PWTT/CCD which use single pre/post features.
# IF uses the full multimodal feature space.
# Parquet: bda_product_prepost.
# =============================================================================
print("=" * 70)
print("CELL E8: ISOLATION FOREST (anomaly detection)")
print("=" * 70)

X, y, groups, feat_names = prepare_Xy(all_feature_cols)
if X is None:
    print("  Insufficient features")
else:
    # contamination = estimated damage rate
    damage_rate = y.mean()
    print(f"  Damage rate: {damage_rate:.3f}")

    for contamination in [damage_rate, 0.05, 0.10]:
        iforest = IsolationForest(contamination=contamination, random_state=RANDOM_STATE, n_jobs=-1)
        iforest.fit(X)
        anomaly_scores = -iforest.score_samples(X)  # higher = more anomalous
        auc = roc_auc_score(y, anomaly_scores)
        preds = (iforest.predict(X) == -1).astype(int)  # -1 = anomaly
        f1 = f1_score(y, preds)
        prec = precision_score(y, preds, zero_division=0)
        rec = recall_score(y, preds, zero_division=0)
        label = f"E8_IsolationForest_c={contamination:.3f}"
        EXPERIMENT_LOG[label] = {'auc': auc, 'f1': f1, 'n_features': X.shape[1], 'method': 'unsupervised'}
        print(f"    c={contamination:.3f}: AUC={auc:.3f}  F1={f1:.3f}  P={prec:.3f}  R={rec:.3f}")
        registry.log_experiment(
            cell_id='cell_e8', experiment_name=label,
            parquet_name='bda_product_prepost', feature_set_name='all_multimodal',
            classifier_name=f'IsolationForest_c{contamination:.3f}',
            feature_cols=feat_names, cv_method='unsupervised', n_folds=0,
            imputation='median', y_true=y, y_proba=anomaly_scores, groups=groups,
            note=f'Isolation Forest anomaly detection, contamination={contamination:.3f}',
            tags=['unsupervised', 'isolation_forest'],
        )

    print(f"\n  IF captures multivariate anomaly patterns (vs NB09b single-feature thresholds)")



# CELL E9: COMPARISON TABLE

In [ ]:
# @title CELL E9: COMPARISON TABLE (all NB09c methods)
print("=" * 70)
print("CELL E9: NB09c COMPARISON TABLE")
print("=" * 70)

print(f"\n  {'Method':<50s} {'AUC':>7s} {'F1':>7s} {'N_feat':>7s}")
print(f"  {'-'*50} {'-'*7} {'-'*7} {'-'*7}")

for key in sorted(EXPERIMENT_LOG.keys()):
    res = EXPERIMENT_LOG[key]
    auc = res.get('auc', np.nan)
    f1 = res.get('f1', np.nan)
    nf = res.get('n_features', '?')
    auc_s = f"{auc:.3f}" if not np.isnan(auc) else "--"
    f1_s = f"{f1:.3f}" if not np.isnan(f1) else "--"
    print(f"  {key:<50s} {auc_s:>7s} {f1_s:>7s} {str(nf):>7s}")

# group analysis
print(f"\n  By category:")
for prefix, label in [('E1_', 'Stacking'), ('E2_', 'LightGBM'), ('E5_', 'Native NaN'),
                       ('E6_A', 'Impute+obs'), ('E6_B', 'NativeNaN'), ('E6_C', 'SAR-only'),
                       ('E7_', 'Non-tree'), ('E8_', 'IsolationForest')]:
    group = {k: v for k, v in EXPERIMENT_LOG.items() if k.startswith(prefix)}
    if group:
        best = max(group.items(), key=lambda x: x[1].get('auc', 0))
        print(f"    {label:20s}: best={best[0]} AUC={best[1].get('auc', 0):.3f}")



# save comparison table
comp_rows = []
for key in sorted(EXPERIMENT_LOG.keys()):
    res = EXPERIMENT_LOG[key]
    comp_rows.append({'method': key, 'auc': res.get('auc', np.nan), 'f1': res.get('f1', np.nan),
                      'n_features': res.get('n_features', 0), 'auc_std': res.get('auc_std', np.nan)})
if comp_rows:
    comp_df = pd.DataFrame(comp_rows).sort_values('auc', ascending=False)
    save_result(comp_df, 'comparison_table', 'cell_e9')

    # bar chart
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, max(4, len(comp_df) * 0.35)))
    ax.barh(comp_df['method'], comp_df['auc'].fillna(0), color='steelblue', edgecolor='black')
    ax.axvline(0.813, color='red', linestyle='--', alpha=0.5, label='Dietrich AUC=0.813')
    ax.set_xlabel('AUC (GroupKFold)')
    ax.set_xlim(0.4, 1.0)
    ax.set_title('NB09c: All Experiments AUC')
    ax.legend()
    save_fig(fig, 'comparison_bar', 'cell_e9')


# CELL E10: SAVE RESULTS

In [ ]:
# @title CELL E10: SAVE OOF PREDICTIONS + RESULTS
import pickle

print("=" * 70)
print("CELL E10: SAVE RESULTS")
print("=" * 70)

# OUT_DIR already set in S0

# save experiment log
results_clean = {}
for key, res in EXPERIMENT_LOG.items():
    results_clean[key] = {k: v for k, v in res.items() if k not in ('y_true', 'y_proba', 'fold_aucs')}
    results_clean[key]['auc'] = float(res.get('auc', 0))
    results_clean[key]['f1'] = float(res.get('f1', 0))

save_result(results_clean, 'experiment_log', 'cell_e10', fmt='json')

# save OOF predictions for NB10 stacking
oof_data = {}
for key, res in EXPERIMENT_LOG.items():
    if 'y_true' in res and 'y_proba' in res:
        oof_data[key] = {'y_true': res['y_true'], 'y_proba': res['y_proba']}

with open(OUT_DIR / "nb09c_oof.pkl", 'wb') as f:
    pickle.dump(oof_data, f)

# summary CSV
summary_rows = []
for name, res in sorted(EXPERIMENT_LOG.items(), key=lambda x: -x[1].get('auc', 0)):
    summary_rows.append({'experiment': name, 'auc': res.get('auc', 0), 'f1': res.get('f1', 0),
                         'n_features': res.get('n_features', 0)})
if summary_rows:
    save_result(pd.DataFrame(summary_rows), 'experiment_summary', 'cell_e10')

print(f"  OOF predictions: {OUT_DIR / 'nb09c_oof.pkl'}")
print(f"  Experiments saved: {len(results_clean)}")



In [ ]:
# @title CELL 19: SAVE REGISTRY
registry.save()
